# RSO-113: Compare thermal behavior across different louver configurations

Check whether different louver configurations lead to noticeable differences in how quickly or how strongly the dome and telescope temperatures respond to outside conditions.



Expected results:

**Plots**: temperature vs time with configuration highlighted

**Comparison tables**: how much temperatures change for each configuration

**Variation view**: results grouped by similar wind conditions (to avoid mixing very different nights)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time
import matplotlib.dates as mdates
import matplotlib.cm as cm
import seaborn as sns

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [2]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

# Queries

In [3]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [4]:
def query_essTemperature(start, end):
    df_esstemperature = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature",
        columns=['location', 'private_identity','sensorName', 'temperatureItem0','timestamp'],
        begin=start,
        end=end,
    )

    return df_esstemperature

# Plot functions

# Configuration of louvers

In [5]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [6]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [7]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

Number of unique configurations detected: 15

Configuration 1 (appears 17 times):
  position2: 100
  position11: 100
  position12: 100
  position20: 100
  position21: 100
  position29: 100
----------------------------------------
Configuration 2 (appears 14 times):
  position2: 10
  position11: 10
  position12: 10
  position20: 10
  position21: 10
  position29: 10
----------------------------------------
Configuration 3 (appears 5 times):
  position2: 50
  position11: 50
  position12: 50
  position20: 50
  position21: 50
  position29: 50
----------------------------------------
Configuration 4 (appears 4 times):
  position11: 100
  position21: 100
  position29: 100
----------------------------------------
Configuration 5 (appears 4 times):
  position2: 100
  position11: 100
  position12: 100
----------------------------------------
Configuration 6 (appears 4 times):
  position20: 100
  position21: 100
  position29: 100
----------------------------------------
Configuration 7 (appears 2

# Prepare the data

In [8]:
df_esstemperature = query_essTemperature(t_start_period, t_end_period)

"ESS:111": "Temperature dome inside",
"ESS:301": "Temperature ess weather station",
"ESS:113": "Temperature m1m3 inside",
"ESS:112": "Temperature m2 inside"
    

In [9]:
# Keep only the ESS identities with 111 and 301
df_temp = df_esstemperature[
    df_esstemperature['private_identity'].isin(['ESS:111', 'ESS:112', 'ESS:113', 'ESS:301'])
].copy()

In [10]:
# Add in df_setlouvers configuration the end of each configuration
df_setlouvers['time_end'] = df_setlouvers['time_stamp'].shift(-1)
df_temp['time'] = pd.to_datetime(df_temp['timestamp'], unit='s')
max_time = df_temp['time'].max()
df_setlouvers['time_end'] = df_setlouvers['time_end'].fillna(max_time)

In [11]:
# Column positions
position_cols = df_setlouvers.filter(regex=r'^position').columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

# Function to labels
def build_conf_label(row):
    config = row[position_cols]
    non_zero = config[config != 0]

    if len(non_zero) == 0:
        return f"Configuration {row['louvers_conf']}: all closed"

    lines = [f"Configuration {row['louvers_conf']}:"]
    for col, val in non_zero.items():
        lines.append(f"{col}: {int(val)}")
    return "\n".join(lines)

df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

In [12]:
# all dates in the same format
df_temp["time"] = pd.to_datetime(df_temp["time"], utc=True)

df_setlouvers["time_stamp"] = pd.to_datetime(df_setlouvers["time_stamp"], utc=True)
df_setlouvers["time_end"] = pd.to_datetime(df_setlouvers["time_end"], utc=True)

In [ ]:
# Sort times
df_temp = df_temp.sort_values("time")
df_setlouvers = df_setlouvers.sort_values("time_stamp")

In [ ]:
df_temp_merge = pd.merge_asof(
    df_temp,
    df_setlouvers,
    left_on="time",
    right_on="time_stamp",
    direction="backward"
)

In [ ]:
df_temp_merge = df_temp_merge[
    df_temp_merge["time"] <= df_temp_merge["time_end"]
]

In [ ]:
df_temp_merge

# temperature vs time with configuration highlighted

Since I didn't know which one was the “highlighted configuration” I set it up so that I could view the graphs with all

In [ ]:
for _, row in df_setlouvers.iterrows():
    conf = row["conf_label"]
    t0 = row["time_stamp"]
    t1 = row["time_end"]
    
    df_interval = df_temp_merge[
        (df_temp_merge["time"] >= t0) &
        (df_temp_merge["time"] <= t1)
    ]
    
    if df_interval.empty:
        continueshow()
    
    plt.figure()
    
    for sensor in df_interval["private_identity_x"].unique():
        df_s = df_interval[df_interval["private_identity_x"] == sensor]
        plt.plot(df_s["time"], df_s["temperatureItem0"], label=sensor)
    
    plt.title(f"{conf} \n {t0} → {t1}")
    plt.xlabel("Time")
    plt.ylabel("Temperature")
    plt.legend()
    plt.show()